# Apply A Chirp Waveform To The Pulse PhMod Template


## 1. Parameters

Edit these values for a different chirp. The TNMR `freq_offset` is derived as `SWEEP_WIDTH_HZ / 2`.


In [ ]:
POINTS = 20000
DWELL_US = 0.2
SWEEP_WIDTH_HZ = -160.0e3


## 2. Setup


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt

repo_search_start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (repo_search_start, *repo_search_start.parents)
    if (candidate / "hardware_capture" / "rf" / "redstone" / "sequence.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
OUTPUT_DIR = REDSTONE_DIR / "examples" / "generated" / "pulse_phmod"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_TPS_PATH = REDSTONE_DIR / "20260630_pulse_phmod_template.tps"
TEMPLATE_TNT_PATH = REDSTONE_DIR / "20260630_pulse_phmod_template.tnt"
OUTPUT_STEM = "20260630_pulse_phmod_chirp_waveform"
OUTPUT_TPS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps"
OUTPUT_TNT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tnt"
OUTPUT_JSON_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps.json"
OUTPUT_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_phase_table.txt"

TARGET_PHASE_TABLE = "phmod_pulse"
TARGET_PHASE_LINE = "F1_PhMod"
TARGET_PHASE_EVENTS = (3, 4)

from hardware_capture.rf.redstone import Chirp, Sequence

assert TEMPLATE_TPS_PATH.exists(), TEMPLATE_TPS_PATH
assert TEMPLATE_TNT_PATH.exists(), TEMPLATE_TNT_PATH
assert OUTPUT_TPS_PATH.resolve() != TEMPLATE_TPS_PATH.resolve()
assert OUTPUT_TNT_PATH.resolve() != TEMPLATE_TNT_PATH.resolve()


## 3. Load And Inspect The Sequence


In [ ]:
sequence = Sequence.load(TEMPLATE_TPS_PATH)
parameter_values = sequence.parameter_values()
target_references = [
    reference
    for reference in sequence.table_references()
    if reference.table_name == TARGET_PHASE_TABLE
]

target_table = next(table for table in sequence.tables if table.get("name") == TARGET_PHASE_TABLE)

{
    "summary": sequence.summary().to_dict(),
    "target_table_rows_before": len((target_table.get("entries") or "").splitlines()),
    "target_references": target_references,
    "parameters": parameter_values,
}


## 4. Build The Chirp


In [ ]:
phase_waveform = Chirp(
    points=POINTS,
    sweep_width_hz=SWEEP_WIDTH_HZ,
    dwell_s=DWELL_US * 1.0e-6,
    offset_hz=-SWEEP_WIDTH_HZ / 2.0,
    phase=0.0,
    phase_steps=2**16,
)

t_mod_offset_text = str(parameter_values["t_mod_offset"]).strip().lower()
if t_mod_offset_text.endswith("ns"):
    t_mod_offset_us = float(t_mod_offset_text[:-2]) * 1.0e-3
elif t_mod_offset_text.endswith("n"):
    t_mod_offset_us = float(t_mod_offset_text[:-1]) * 1.0e-3
elif t_mod_offset_text.endswith("us"):
    t_mod_offset_us = float(t_mod_offset_text[:-2])
elif t_mod_offset_text.endswith("u"):
    t_mod_offset_us = float(t_mod_offset_text[:-1])
else:
    raise ValueError(f"Expected t_mod_offset in ns or us, got {t_mod_offset_text!r}.")

duration_us = POINTS * DWELL_US
duration_ms = duration_us * 1.0e-3
t_pulse_us = duration_us - t_mod_offset_us
if t_pulse_us <= 0:
    raise ValueError("t_pulse must be positive after subtracting t_mod_offset.")

freq_offset_hz = SWEEP_WIDTH_HZ / 2.0
frequency_hz = [value - phase_waveform.offset_hz for value in phase_waveform.frequency_values()]

{
    "duration_us": duration_us,
    "t_mod_offset_us": t_mod_offset_us,
    "t_pulse_us": t_pulse_us,
    "freq_offset_hz": freq_offset_hz,
    "phase_mod_frequency_span_hz": (frequency_hz[0], frequency_hz[-1]),
    "waveform": phase_waveform.summary().to_dict(),
}


## 5. Visualize The Chirp Pulse


In [ ]:
time_ms = [index * phase_waveform.dwell_s * 1.0e3 for index in range(POINTS)]
phase_deg = phase_waveform.phase_values()
if phase_deg is None:
    raise ValueError("Phase output is not available.")

envelope_i = [math.cos(math.radians(value)) for value in phase_deg]
envelope_q = [math.sin(math.radians(value)) for value in phase_deg]
plot_step = max(1, POINTS // 5000)
plot_indices = list(range(0, POINTS, plot_step))
if plot_indices[-1] != POINTS - 1:
    plot_indices.append(POINTS - 1)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot([time_ms[i] for i in plot_indices], [frequency_hz[i] * 1.0e-3 for i in plot_indices])
axes[0].set_ylabel("Phase mod (kHz)")
axes[0].set_title(f"{TARGET_PHASE_TABLE}: {duration_ms:.3f} ms chirp")
axes[0].grid(True, alpha=0.3)

axes[1].plot([time_ms[i] for i in plot_indices], [phase_deg[i] for i in plot_indices])
axes[1].set_ylabel("Phase (deg)")
axes[1].set_ylim(-5, 365)
axes[1].grid(True, alpha=0.3)

axes[2].plot([time_ms[i] for i in plot_indices], [envelope_i[i] for i in plot_indices], label="I")
axes[2].plot([time_ms[i] for i in plot_indices], [envelope_q[i] for i in plot_indices], label="Q")
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("Unit envelope")
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc="upper right")

fig.tight_layout()
plt.show()


## 6. Apply And Save The Sequence


In [ ]:
phase_rows = phase_waveform.phase_column(precision=3)
if phase_rows is None:
    raise ValueError("Phase output is not available.")

sequence.set_waveform_to_table(
    TARGET_PHASE_TABLE,
    phase_waveform,
    quantity="phase",
    precision=3,
    phase_units="deg",
    allow_shared=True,
)
for event_number in TARGET_PHASE_EVENTS:
    sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, TARGET_PHASE_TABLE)

sequence.set_parameter_value("freq_offset", f"{freq_offset_hz:.15g}")
sequence.set_parameter_value("t_pulse", f"{t_pulse_us:.15g}u")

phase_waveform.write_phase_column(OUTPUT_PHASE_TXT_PATH, precision=3)
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=False)
sequence.write_tps(OUTPUT_TPS_PATH)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_table = next(table for table in reloaded.tables if table.get("name") == TARGET_PHASE_TABLE)
assert len((reloaded_table.get("entries") or "").splitlines()) == POINTS
assert reloaded.parameter_values()["freq_offset"] == f"{freq_offset_hz:.15g}"
assert reloaded.parameter_values()["t_pulse"] == f"{t_pulse_us:.15g}u"

{
    "phase_rows": len(phase_rows),
    "first_10": phase_rows[:10],
    "last_10": phase_rows[-10:],
    "output_tps": OUTPUT_TPS_PATH,
    "output_json": OUTPUT_JSON_PATH,
    "output_phase_table": OUTPUT_PHASE_TXT_PATH,
}


## 7. Save The TNT Copy


In [ ]:
tps_signature = b"PSEQ1.001.18 BIN"
tnt_embedded_signature = b"PSEQ" + (1).to_bytes(4, "little") + b"1.18 BIN"

template_tps_bytes = TEMPLATE_TPS_PATH.read_bytes()
edited_tps_bytes = OUTPUT_TPS_PATH.read_bytes()
template_tnt_bytes = TEMPLATE_TNT_PATH.read_bytes()

if not template_tps_bytes.startswith(tps_signature):
    raise ValueError("Template TPS has an unexpected PSEQ signature.")
if not edited_tps_bytes.startswith(tps_signature):
    raise ValueError("Edited TPS has an unexpected PSEQ signature.")

template_embedded = tnt_embedded_signature + template_tps_bytes[16:]
edited_embedded = tnt_embedded_signature + edited_tps_bytes[16:]
start = template_tnt_bytes.find(template_embedded)
if start < 0:
    raise ValueError("The template TNT does not contain the sibling template TPS payload.")

OUTPUT_TNT_PATH.write_bytes(
    template_tnt_bytes[:start]
    + edited_embedded
    + template_tnt_bytes[start + len(template_embedded) :]
)

restored_embedded = OUTPUT_TNT_PATH.read_bytes()[start : start + len(edited_embedded)]
assert tps_signature + restored_embedded[16:] == edited_tps_bytes

{
    "output_tnt": OUTPUT_TNT_PATH,
    "tnt_bytes": OUTPUT_TNT_PATH.stat().st_size,
    "embedded_tps_offset": start,
}


## 8. Output Summary


In [ ]:
generated_files = [
    OUTPUT_PHASE_TXT_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_TPS_PATH,
    OUTPUT_TNT_PATH,
]

[
    {"file": path.name, "path": str(path), "bytes": path.stat().st_size}
    for path in generated_files
]
